# Phi RDS 设计走读

对着 [`docs/DESIGN.md`](../docs/DESIGN.md) 跑一遍。目标：

1. 把关键 **Python 语法**（dataclass / frozen / `|` / Literal）捡回来  
2. 亲手摸一遍 **LazyHandle → Body → Episode → Layout → Schema → 写盘 → 打开**

```bash
cd phi_rds && pip install -e ".[lerobot]" jupyter
jupyter notebook examples/understand_phi_rds.ipynb
```

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# 保证能 import 到 src 布局的包（未 pip install -e 时）
ROOT = Path.cwd()
if (ROOT / "src" / "phi_rds").is_dir():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src" / "phi_rds").is_dir():
    sys.path.insert(0, str(ROOT.parent / "src"))

import numpy as np
import phi_rds
print("phi_rds", phi_rds.__version__, "@", Path(phi_rds.__file__).parent)

phi_rds 0.1.0 @ /Users/rookie/Desktop/EAI/phi_data/phi_rds/src/phi_rds


---
## 1. 语法热身：dataclass / frozen / replace / `|` / Literal

Phi RDS 几乎全是 `@dataclass`。先用玩具例子回忆语法，再对照真实类型。

In [ ]:
from dataclasses import dataclass, field, replace
from typing import Literal

# --- 可变 dataclass：字段可以原地改 ---
@dataclass
class Box:
    name: str
    tags: list[str] = field(default_factory=list)  # 可变默认值必须用 factory

b = Box("a")
b.tags.append("x")
print("mutable:", b)

# --- frozen：创建后不能赋值，改用 replace 出新对象 ---
@dataclass(frozen=True)
class Point:
    x: float
    y: float = 0.0

p = Point(1.0)
try:
    p.x = 2.0  # noqa: intentional
except Exception as e:
    print("frozen assign →", type(e).__name__, e)

p2 = replace(p, x=2.0)  # 拷贝改字段
print("replace:", p, "→", p2)

# --- Python 3.10+ 联合类型写法：A | B 等价 Optional / Union ---
Level = Literal["L1", "L2"]  # 只能取这两个字符串之一

def stamp(level: Level | None = None) -> str:
    return level or "(none)"

print("Literal:", stamp("L2"), stamp(None))

**对照 Phi RDS**

| 语法 | 用在哪 |
|------|--------|
| `@dataclass` 可变 | `Episode`, `LazyHandle`, `RobotDataSet`（有缓存） |
| `@dataclass(frozen=True)` | `DatasetMeta`, `Slice`, `Layout`, `DatasetSchema`… |
| `replace(...)` | `Episode.with_meta`, `DatasetMeta.bind_schema`, 视图 |
| `A \| None` | 几乎所有可选字段 |
| `Literal[...]` | `DataLevel`, `EpisodeStatus`, `StatsScope`（`types.py`） |
| `field(default_factory=...)` | 默认 `dict`/`list`/嵌套 dataclass |
| `Protocol` | `EpisodeStore`：只要有 `open/list_metas/exists` 即可 |

---
## 2. LazyHandle：先挂引用，用时再 `.load()`

几何 / 相机默认不立刻把大数组塞进 Episode。

In [ ]:
from phi_rds import LazyHandle
from phi_rds.core.lazy import StorageKind

arr = np.arange(6, dtype=np.float64).reshape(2, 3)
h = LazyHandle.from_memory(arr)

print("kind:", h.storage.kind)          # StorageKind.MEMORY
print("shape/dtype:", h.shape, h.dtype)
print("load() 同一对象:", h.load() is arr)

# 视频：只记路径，不读像素
v = LazyHandle.from_video("/fake/cam.mp4", from_timestamp=0.0, fps=30.0)
print("video kind:", v.storage.kind)
print("video path:", v.storage.video.path)
print("还没有 cache:", v._cache is None)

# 自定义 loader：第一次 load 才调用
from phi_rds.core.lazy import StorageRef

calls = {"n": 0}

def slow_loader(handle):
    calls["n"] += 1
    return np.ones(3) * calls["n"]

lazy = LazyHandle(storage=StorageRef(kind=StorageKind.OTHER)).bind_loader(slow_loader)
print("1st", lazy.load(), "calls", calls["n"])
print("2nd", lazy.load(), "calls", calls["n"], "← 走 cache")

kind: StorageKind.MEMORY
shape/dtype: (2, 3) float64
load() 同一对象: True
video kind: StorageKind.VIDEO_MP4
video path: /fake/cam.mp4
还没有 cache: True
1st [1. 1. 1.] calls 1
2nd [1. 1. 1.] calls 1 ← 走 cache


### 2.1 LeRobot v3 视频怎么存、怎么加载

**不是**「一集一个 mp4」。v3 常见是：

```text
videos/observation.images.wrist/chunk-000/file-000.mp4   ← 一个文件里可有多集
meta/episodes/... 每集一行：
  videos/.../chunk_index, file_index
  videos/.../from_timestamp, to_timestamp   ← 本集在该文件内的时间窗（秒）
```

官方取第 `ep` 集、相对本集 `t` 秒的帧：

1. `path = meta.get_video_file_path(ep, video_key)`
2. `abs_t = from_timestamp + t`
3. `decode_video_frames(path, [abs_t], …)` seek 解码

我们打开数据集时只把 `{path, from, to}` 挂进 `VideoLocator`，**不解码**；训练看图用 `ds.as_lerobot()`。


In [ ]:
from phi_rds.core.lazy import VideoLocator, StorageRef, StorageKind

# 模拟：同一 mp4，两集各占一段（秒）
shared = "/data/ds/videos/observation.images.wrist/chunk-000/file-000.mp4"
ep0 = VideoLocator(path=shared, from_timestamp=0.0, to_timestamp=2.5, fps=30.0)
ep1 = VideoLocator(path=shared, from_timestamp=2.5, to_timestamp=5.0, fps=30.0)
print("same file?", ep0.path == ep1.path)
print("ep0 window:", ep0.from_timestamp, "→", ep0.to_timestamp)
print("ep1 window:", ep1.from_timestamp, "→", ep1.to_timestamp)

# 官方加载公式（相对本集 0.1s 的那一帧）
t_rel = 0.1
print("decode seek ep0:", ep0.from_timestamp + t_rel, "s in", ep0.path)
print("decode seek ep1:", ep1.from_timestamp + t_rel, "s in", ep1.path)

# SDK 挂句柄（与 LeRobotEpisodeStore._load_cameras 一致）
h0 = LazyHandle(
    storage=StorageRef(kind=StorageKind.VIDEO_MP4, uri=shared, video=ep0),
    dtype="uint8",
)
print("handle uri:", h0.storage.uri)
print("handle segment:", h0.storage.video)
print("no pixels until as_lerobot() / external decode; cache=", h0._cache)


`from_arrays(cameras={名: 路径})` 只是**写之前**的简写（源往往是一集一段文件）。  
读标准 v3 时以 episodes 表的 **file_index + from/to_timestamp** 为准；解码交给官方 `as_lerobot()`。


In [ ]:
from phi_rds import keys

Episode(
    uid="ep-demo",
    fps=30.0,
    task="pick",
    cameras={"wrist": str(demo_mp4)},
)

'back_left_fisheye'

**要不要在库里实现视频解码？建议：核心先不要。**

| 做法 | 评价 |
|------|------|
| 核心 `LazyHandle.load()` 内置解码 | ❌ 拖进 `av`/`opencv` 重依赖；和「句柄默认不物化」冲突 |
| `io` 提供可选 `bind_video_loader(handle)`（extra deps） | ✅ 以后有质检/可视化需求再加 |
| 训练像素 | ✅ 已有旁路：`ds.as_lerobot()`，官方管解码 |
| Notebook / 业务侧 `bind_loader` | ✅ 现在这样就够演示与定制 |

SDK 的职责是：**记住路径 + 时间窗 + 何时物化**；用哪套解码器留给调用方或可选插件。

---
## 3. 几何树：Pose → Arm → RobotBody

`Pose` **可存任意表示**，用 `rep` 标明（默认 `matrix` = `(N,4,4)`）。  
需要齐次矩阵时 `as_matrix()`；需要向量写法时 `as_rep("pose9")` 等。

In [3]:
from phi_rds import Pose, Arm, Gripper, JointGroup, RobotBody, keys

N = 4
T = np.tile(np.eye(4), (N, 1, 1))
T[:, 0, 3] = np.linspace(0, 0.3, N)  # 沿 x 平移

pose = Pose.from_matrix(T)  # rep="matrix"
print("rep / values:", pose.rep, pose.numpy().shape)
print("as pose9:", pose.as_pose9().shape, "← (N,9) = xyz + 旋转前两列")
print("x:", pose.as_matrix()[:, 0, 3])

# 也可以原样存四元数，不必先转矩阵
qpose = Pose.from_rep(pose.as_quat(), "quat_wxyz")
print("quat storage:", qpose.rep, qpose.numpy().shape, "→ matrix", qpose.as_matrix().shape)

joints = JointGroup(
    position=LazyHandle.from_memory(np.zeros((N, 7))),
    names=keys.arm_joint_names(keys.ARM_SINGLE, 7),
)
grip = Gripper(open=LazyHandle.from_memory(np.linspace(0, 1, N).reshape(N, 1)))

body = RobotBody(
    arm_single=Arm(ee=pose, joints=joints, gripper=grip),
)
print("ee_matrix:", body.ee_matrix(keys.ARM_SINGLE).shape)
print("joint names:", body.joint_group(keys.ARM_SINGLE).names[:3], "...")
print("gripper[0]:", body.gripper_of(keys.ARM_SINGLE)[0])

matrix shape: (4, 4, 4)
as pose9: (4, 9) ← (N,9) = xyz + 旋转前两列
x: [0.  0.1 0.2 0.3]
ee_matrix: (4, 4, 4)
joint names: ('joint1', 'joint2', 'joint3') ...
gripper[0]: [0.]


---
## 4. Episode：一条轨迹 = Meta + state/action + 相机 + 标签

入门用 `Episode.from_arrays`；不必手搓 `EpisodeMeta`。

In [ ]:
from phi_rds import Episode
from dataclasses import asdict, is_dataclass

ep = Episode.from_arrays(
    "ep-demo",
    fps=30.0,
    task="pick_cup",
    ee=T,
    ee_action=T.copy(),
    joints=np.zeros((N, 7)),
    joints_action=np.zeros((N, 7)),
    gripper=np.linspace(0.2, 0.9, N),
    gripper_action=np.linspace(0.2, 0.9, N),
)

print("uid / frames / task:", ep.uid, ep.num_frames, ep.labels.task)
print("meta:", ep.meta)
print("state.arm_single.ee:", ep.state.arm_single.ee.numpy().shape)
print("action is None?", ep.action is None)
print("fields 默认空:", len(ep.fields), "← 主路径用几何树，不是 FieldMap")

# frozen EpisodeMeta 要改字段 → with_meta / replace
ep2 = ep.with_meta(task="place_cup")
print("with_meta:", ep.meta.task, "→", ep2.meta.task)

---
## 5. Layout / Slice：几何树 ↔ 扁平列

`Slice` 说清楚：**从 state 还是 action、哪个 part、什么 kind**。  
`Feature` 把多段 Slice 横拼成一列 LeRobot feature。  
`Layout` = 一整套 Feature。

In [ ]:
from phi_rds import preset_layout, encode, decode, Slice, Feature, Layout

layout = preset_layout("umi", arms="single")
print("layout.id:", layout.id, "robot:", layout.robot_type)
for f in layout.features:
    kinds = [s.kind for s in f.slices]
    print(f"  {f.name:40s}  slices={kinds}  width≈{sum(len(s.names) for s in f.slices)}")

# 看一个 Slice 长什么样（frozen dataclass）
s0 = layout.features[0].slices[0]
print("\nfirst Slice:")
print(" ", s0)
print("  kind / from_ / part / pose_rep:", s0.kind, s0.from_, s0.part, s0.pose_rep)
print("  resolved_pose_rep:", s0.resolved_pose_rep)

In [ ]:
# encode：Episode → 列 dict；decode：列 → Decoded → 再拼回 Body（io 层做）
cols = encode(ep, layout)
for name, arr in cols.items():
    print(f"{name:40s} {np.asarray(arr).shape}")

decoded = decode(cols, layout)
print("\nDecoded.poses keys:", list(decoded.poses.keys()))
key = ("state", keys.ARM_SINGLE)
print("pose shape:", decoded.poses[key].shape)
print("roundtrip x:", decoded.poses[key][:, 0, 3])
assert np.allclose(decoded.poses[key][:, 0, 3], T[:, 0, 3])

手拼一个最小 Layout（理解 `Slice` 字段）：

In [ ]:
tiny = Layout(
    id="tiny",
    robot_type="umi",
    frame=keys.BASE,
    features=(
        Feature(
            keys.STATE_ARM_TCP,
            (
                Slice(
                    kind="pose9",           # 或 kind="pose" + pose_rep="pose9"
                    from_="state",          # 注意字段名是 from_（from 是关键字）
                    part=keys.ARM_SINGLE,
                    names=keys.pose9_names(keys.ARM_SINGLE),
                ),
            ),
        ),
    ),
)
print(encode(ep, tiny)[keys.STATE_ARM_TCP].shape)  # (N, 9)

---
## 6. DatasetSchema：整集契约 + L1/L2 stamp

- `Layout`：Episode↔列怎么绑  
- `FeatureSpec`：给官方 `LeRobotDataset.create(features=...)` 的 dtype/shape  
- `DatasetSchema`：两者打成一包，并带 `level` / `standard_version`

In [ ]:
from phi_rds import schema_for, DatasetSchema, LATEST_STANDARD, resolve_standard

s1 = schema_for("umi", level="L1")
s2 = schema_for("umi", level="L2")

print("L1:", s1.level, "standard=", s1.standard_version, "profile=", s1.profile)
print("L2:", s2.level, "standard=", s2.standard_version, "profile=", s2.profile)
print("LATEST:", LATEST_STANDARD, resolve_standard(None))

print("\nFeatureSpec → LeRobot dict (截一段):")
feat = s2.to_lerobot_features()
for k in list(feat)[:2]:
    print(" ", k, feat[k])

print("\nstamp for info.json:", s2.to_stamp())
assert s2.layout is not None and s2.layout.id == "umi_single"

---
## 7. 端到端：create_l2 → finalize → open_dataset → as_lerobot

门面函数把 `DatasetSchema` + `DatasetMeta` + `DatasetWriter` 藏起来。

In [ ]:
from phi_rds import create_l2, open_dataset, encode_pose, change_frame
import json

out = Path("_nb_demo_l2")

ep_a = Episode.from_arrays(
    "ep-000", fps=30.0, task="pick",
    ee=T, ee_action=T,
    joints=np.zeros((N, 7)), joints_action=np.zeros((N, 7)),
    gripper=np.ones(N), gripper_action=np.ones(N),
)
T2 = T.copy()
T2[:, 1, 3] = 0.05  # 只改平移，别对齐次矩阵做 +标量
ep_b = Episode.from_arrays(
    "ep-001", fps=30.0, task="pick",
    ee=T2, ee_action=T2,
    joints=np.ones((N, 7)) * 0.1, joints_action=np.ones((N, 7)) * 0.1,
    gripper=np.zeros(N), gripper_action=np.zeros(N),
)

w = create_l2(out, robot="umi", name="nb_demo", version="v1")
w.add_episode(ep_a)
w.add_episode(ep_b)
root = w.finalize()
print("wrote", root)
print("schema", w.schema.level, w.schema.standard_version)

info = json.loads((root / "meta" / "info.json").read_text())
print("phi stamps:", {k: info[k] for k in info if k.startswith("phi_")})
print("uids:", json.loads((root / "meta" / "episode_uids.json").read_text()))

In [ ]:
ds = open_dataset(root)  # 按 robot_type 自动套 umi_single layout
print("len / level / standard:", len(ds), ds.level, ds.standard_version)
print("index rows:", list(ds.index))

got = ds[0]
print("opened:", got.uid, got.num_frames, got.labels.task)
print("ee x:", got.state.arm_single.ee.numpy()[:, 0, 3])

# 视图：不复制磁盘，只换 EpisodeIndex
view = ds.select(["ep-001"])
print("view:", len(view), "is_view", view.meta.view.is_view, view.meta.view.selector)
print("view[0].uid", view[0].uid)

# 训练旁路
official = ds.as_lerobot()
print("LeRobotDataset frames/fps:", official.meta.total_frames, official.meta.fps)

---
## 8. convert：位姿表示 / 换系

`PoseRep` 描述一种向量写法；`encode_pose` / `change_frame` 是纯函数，不碰磁盘。

In [ ]:
from phi_rds import encode_pose, decode_pose, change_frame, invert_T, lookup_T, FrameGraph, FrameEdge
from phi_rds.convert.pose import PRESETS, resolve_rep

print("可用 PoseRep:", list(PRESETS))
print("pose9 dim:", resolve_rep("pose9").dim, resolve_rep("pose9").names)

M = got.state.arm_single.ee.numpy()
p9 = encode_pose(M, "pose9")
back = decode_pose(p9, "pose9")
print("roundtrip ok:", np.allclose(M, back, atol=1e-6))

# 换系：左乘 T_B_A
T_B_A = np.eye(4)
T_B_A[:3, 3] = [0.1, 0, 0.05]
Mb = change_frame(M, T_B_A)
print("x0 before/after:", M[0, 0, 3], Mb[0, 0, 3])

# FrameGraph + lookup_T
T_base_wrist = np.eye(4)
T_base_wrist[:3, 3] = [0.5, 0, 0.3]
g = FrameGraph(
    frames=("base", "wrist"),
    edges=(
        FrameEdge(
            child="wrist",
            parent="base",
            matrix_flat16=tuple(T_base_wrist.reshape(-1).tolist()),
        ),
    ),
    root="base",
)
print("lookup wrist→base:\n", lookup_T(g, "wrist", "base"))
print("inv check:", np.allclose(invert_T(T_base_wrist) @ T_base_wrist, np.eye(4)))

---
## 9. 对象关系速查（跑完后对照）

```text
create_l2 / schema_for
        → DatasetSchema { FeatureSpec*, Layout? }
        → DatasetWriter.add_episode(Episode)
              Episode.state: RobotBody
                arm_single: Arm { ee: Pose(LazyHandle), joints, gripper }
              encode(Layout) → 列 → LeRobot add_frame

open_dataset
        → RobotDataSet { meta, schema, index, store }
        → ds[i] → store.open → decode → Episode
        → ds.as_lerobot() → 官方训练 API
```

| 你忘了的语法 | 在本 notebook 哪节 |
|-------------|-------------------|
| `frozen` / `replace` | §1、§4 `with_meta` |
| `field(default_factory=)` | §1 |
| `X \| None` / `Literal` | §1、`types.DataLevel` |
| `from_` 为啥带下划线 | §5 Slice（`from` 是关键字） |
| 静态工厂 `@staticmethod` | `LazyHandle.from_memory`、`Episode.from_arrays` |
| Protocol 鸭子类型 | `EpisodeStore`（内存 / LeRobot 都是 store） |

更完整的字段表 → **`docs/DESIGN.md`**。

In [ ]:
# 可选清理
# import shutil; shutil.rmtree(out, ignore_errors=True)
print("demo dir:", out.resolve())